# 03 — Nihai eğitim, kilitli test tahminleri ve bağımsız test

Bu notebook iki bölümden oluşur:

* **Bölüm A (etiketsiz):** `02_nested_cv.ipynb` tarafından seçilen nihai konfigürasyon tüm development kümesinde üç ağırlık-başlangıç seed'iyle eğitilir. Modeller, ön işleme durumları ve öğrenilmiş dönüşüm istatistikleri kaydedilir. Test kümesinin **yalnızca girdi özellikleri** okunur ve etiket sütunu olmayan `results/test_predictions_locked.csv` dosyası yazılır.
* **Bölüm B (etiketli):** Yalnızca `test_predictions_locked.csv` dosyası ayrı bir git commit'inde kaydedildikten sonra çalışır. Notebook yeniden çalıştırıldığında Bölüm A tahminleri yeniden üretir ve kilitli dosyayla birebir aynı olduklarını doğrular. Ancak bundan sonra test etiketleri okunur.

**Önceden ilan edilen ana model:** nihai konfigürasyon, birincil ağırlık seed'i **31076** (`SEEDS["torch"]`). Karar eşiği **0.50**'dir. Test sonuçlarına göre model, eşik veya ön işleme değiştirilmez.

In [ ]:
import sys, json, time, hashlib, subprocess, platform, warnings
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, torch, joblib, sklearn
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from sklearn.metrics import roc_curve, precision_recall_curve, brier_score_loss
from src import project as P, nested as N
P.set_global_seeds()
pd.set_option("display.width", 180); pd.set_option("display.max_columns", 40)
MODELS = P.RESULTS / "final_models"; MODELS.mkdir(exist_ok=True)
LOCKED = P.RESULTS / "test_predictions_locked.csv"
print("Python", platform.python_version(), "| torch", torch.__version__, "| sklearn", sklearn.__version__)

## Bölüm A — Nihai eğitim (test etiketleri kullanılmaz)

In [ ]:
sel = P.load_json(P.RESULTS / "final_selection.json")
FINAL = sel["config"]; FINAL_CODE = sel["final_candidate"]
WEIGHT_SEEDS = [P.SEEDS["torch"], P.SEEDS["torch_stability_2"], P.SEEDS["torch_stability_3"]]
PRIMARY_SEED = P.SEEDS["torch"]
print("Nihai aday:", FINAL_CODE, "|", sel["final_method"]); print(json.dumps(FINAL, indent=1))
print("Ağırlık seed'leri:", WEIGHT_SEEDS, "| ana seed:", PRIMARY_SEED)
X_dev, y_dev = P.load_development()
X_test = P.load_test_features()          # yalnızca özellikler; etiket yok
assert "y" not in X_test.columns
print("Development:", X_dev.shape, "| Test özellikleri:", X_test.shape)

In [ ]:
def learned_stats(pre):
    num = pre.named_transformers_["num"]; cat = pre.named_transformers_["cat"]
    out = {"numeric_columns": P.NUMERIC_COLS,
           "imputer_median": dict(zip(P.NUMERIC_COLS, num.named_steps["imputer"].statistics_.tolist()))}
    sc = num.named_steps["scaler"]
    for attr in ["mean_", "scale_", "center_", "data_min_", "data_max_", "min_"]:
        if hasattr(sc, attr) and getattr(sc, attr) is not None:
            out[f"scaler_{attr.rstrip('_')}"] = dict(zip(P.NUMERIC_COLS, np.asarray(getattr(sc, attr)).tolist()))
    if "clipper" in num.named_steps:
        cl = num.named_steps["clipper"]
        out["clip_lower_P1"] = dict(zip(P.NUMERIC_COLS, cl.lower_.tolist()))
        out["clip_upper_P99"] = dict(zip(P.NUMERIC_COLS, cl.upper_.tolist()))
    out["onehot_categories"] = {c: list(map(str, cats)) for c, cats in
                                zip(P.CATEGORICAL_COLS, cat.named_steps["onehot"].categories_)}
    out["feature_names_out"] = list(pre.get_feature_names_out())
    return out

final_models, histories, infos = {}, [], []
for s in WEIGHT_SEEDS:
    m = P.train_mlp(X_dev, y_dev, FINAL, weight_seed=s)
    final_models[s] = m
    torch.save(m.model.state_dict(), MODELS / f"mlp_{FINAL_CODE}_seed{s}.pt")
    joblib.dump(m.preprocessor, MODELS / f"preprocessor_{FINAL_CODE}_seed{s}.joblib")
    P.save_json(learned_stats(m.preprocessor), MODELS / f"learned_transform_stats_seed{s}.json")
    h = m.history.copy(); h["weight_seed"] = s; histories.append(h)
    infos.append({"weight_seed": s, **m.info})
hist = pd.concat(histories, ignore_index=True)
hist.to_csv(P.RESULTS / "logs" / "training_histories_final.csv", index=False)
final_info = pd.DataFrame(infos); final_info.to_csv(P.RESULTS / "final_training_runs.csv", index=False)
final_info

In [ ]:
# Referans modeller: tüm development'ta, sabit kuralla eğitilir
ref_models = {}
for name in ["LogReg", "Dummy"]:
    model, t_fit = N.fit_reference(name, X_dev, y_dev)
    ref_models[name] = model
    joblib.dump(model, MODELS / f"reference_{name}.joblib")
    print(name, "eğitim süresi %.3f s" % t_fit)

pm = final_models[PRIMARY_SEED]
P.save_json({
    "final_candidate": FINAL_CODE, "final_method": sel["final_method"],
    "architecture": {"input_dim": pm.info["n_inputs"], "hidden_layers": FINAL["hidden_layers"],
                     "activation": FINAL["activation"], "dropout": FINAL["dropout"], "output": "1 logit",
                     "loss": "BCEWithLogitsLoss", "probability": "sigmoid(logit)",
                     "n_trainable_parameters": pm.info["n_params"]},
    "optimizer": {"name": FINAL["optimizer"], "initial_lr": FINAL["lr"], "weight_decay": FINAL["weight_decay"],
                  "momentum": 0.9 if FINAL["optimizer"] == "sgd" else None, "batch_size": FINAL["batch_size"]},
    "preprocessing": {"numeric_imputer": "median", "scaler": FINAL["scaler"], "clipping_P1_P99": FINAL["clipping"],
                      "categorical": "'unknown' ayrı kategori; OneHotEncoder(handle_unknown='ignore')",
                      "pdays_rule": "999 -> NaN + pdays_no_prev_contact göstergesi",
                      "learned_stats_files": [f"results/final_models/learned_transform_stats_seed{s}.json" for s in WEIGHT_SEEDS]},
    "training": {k: v for k, v in P.TRAINING.items()}, "scheduler_on": FINAL["scheduler"],
    "monitor_split_seed": P.SEEDS["early_stop_split"],
    "weight_seeds": WEIGHT_SEEDS, "primary_seed": PRIMARY_SEED,
    "epochs": {str(r["weight_seed"]): {"best_epoch": r["best_epoch"], "stopped_epoch": r["stopped_epoch"]} for r in infos},
    "decision_threshold": 0.5,
    "dev_inner_cv_mean_AP": sel["dev_inner_cv_mean_AP"],
}, P.RESULTS / "final_hyperparameters.json")

In [ ]:
# Etiketsiz test tahminleri
pred = pd.DataFrame({"original_index": X_test.index.values})
for s in WEIGHT_SEEDS:
    prob = final_models[s].predict_proba(X_test)
    pred[f"mlp_seed{s}_prob"] = prob
    pred[f"mlp_seed{s}_pred"] = (prob >= 0.5).astype(int)
for name, model in ref_models.items():
    prob = model.predict_proba(X_test)[:, 1]
    pred[f"{name.lower()}_prob"] = prob
    pred[f"{name.lower()}_pred"] = (prob >= 0.5).astype(int)
assert not any(c in pred.columns for c in ["y", "label", "target"])

def is_committed(path):
    try:
        out = subprocess.run(["git", "log", "--format=%H %cI", "--", str(path.relative_to(ROOT))],
                             cwd=ROOT, capture_output=True, text=True, check=True).stdout.strip().splitlines()
        return out[-1] if out else None
    except Exception:
        return None

LOCK_COMMIT = is_committed(LOCKED)
if LOCK_COMMIT is None:
    pred.to_csv(LOCKED, index=False, float_format="%.10f")
    print("Kilitli tahmin dosyası yazıldı:", LOCKED.name, "| sha256:", hashlib.sha256(LOCKED.read_bytes()).hexdigest())
    print("Test etiketleri OKUNMADI. Dosyayı ayrı bir commit ile kaydedip GitHub'a gönderin, sonra notebook'u yeniden çalıştırın.")
else:
    locked = pd.read_csv(LOCKED)
    assert list(locked.columns) == list(pred.columns)
    max_diff = np.max(np.abs(locked.filter(like="_prob").values - pred.filter(like="_prob").values))
    same_cls = (locked.filter(like="_pred").values == pred.filter(like="_pred").values).all()
    print("Kilitli dosyanın ilk commit'i:", LOCK_COMMIT)
    print("Yeniden üretilen olasılıklar ile kilitli olasılıklar arasındaki en büyük fark:", max_diff, "| sınıflar aynı:", same_cls)
    assert max_diff < 1e-8 and same_cls, "Yeniden üretilen tahminler kilitli dosyayla aynı değil!"
    print("sha256:", hashlib.sha256(LOCKED.read_bytes()).hexdigest())

## Bölüm B — Test metrikleri (yalnızca kilitli commit'ten sonra)

Aşağıdaki hücre kilitli dosya git geçmişinde yoksa çalışmayı durdurur. Analizlerde yeniden üretilen değil, **kilitli dosyadaki** tahminler kullanılır.

In [ ]:
if LOCK_COMMIT is None:
    raise RuntimeError("test_predictions_locked.csv henüz commit edilmedi; test etiketleri okunmayacak.")
locked = pd.read_csv(LOCKED).set_index("original_index")
_, y_all = P.prepare_xy(P.load_raw())
y_test = y_all.loc[locked.index]        # test etiketleri ilk kez burada okunur
print("Test n =", len(y_test), "| pozitif:", int(y_test.sum()), "(%.2f%%)" % (100 * y_test.mean()))

In [ ]:
models = {f"MLP {FINAL_CODE} seed {s}" + (" (ana)" if s == PRIMARY_SEED else ""): f"mlp_seed{s}" for s in WEIGHT_SEEDS}
models.update({"Logistic Regression": "logreg", "Dummy (most_frequent)": "dummy"})
rows = []
for label, key in models.items():
    r = P.compute_metrics(y_test, locked[f"{key}_prob"])
    r["Brier"] = brier_score_loss(y_test, locked[f"{key}_prob"])
    rows.append({"model": label, **r})
test_metrics = pd.DataFrame(rows).set_index("model")
test_metrics.to_csv(P.RESULTS / "test_metrics.csv")
test_metrics.round(4)

In [ ]:
seed_rows = test_metrics.iloc[:3].copy(); seed_rows.insert(0, "weight_seed", WEIGHT_SEEDS)
mcols = P.METRIC_NAMES
summary = pd.DataFrame([seed_rows[mcols].mean(), seed_rows[mcols].std(ddof=1)], index=["mean", "sd"])
seed_tab = pd.concat([seed_rows[["weight_seed"] + mcols], summary])
seed_tab.to_csv(P.RESULTS / "seed_stability_results.csv")
print("Üç başlangıç seed'i (betimsel; ana sonuç önceden ilan edilen seed 31076'dır)")
display(seed_tab.round(4))
print({m: P.mean_sd(seed_rows[m]) for m in mcols})

### Bootstrap %95 yüzdelik güven aralıkları (1.000 tekrar)

In [ ]:
def bootstrap_ci(y, p, n_boot=1000, seed=P.SEEDS["bootstrap"]):
    rng = np.random.default_rng(seed)
    y = np.asarray(y); p = np.asarray(p); n = len(y)
    vals, rejected = [], 0
    while len(vals) < n_boot:
        idx = rng.integers(0, n, n)
        if len(np.unique(y[idx])) < 2:   # iki sınıf da bulunmalı
            rejected += 1; continue
        m = P.compute_metrics(y[idx], p[idx])
        vals.append((m["ROC_AUC"], m["AP"], m["F1"]))
    vals = np.array(vals)
    return vals, rejected

ci_rows, boot_store = [], {}
for label, key in [(list(models)[0], f"mlp_seed{PRIMARY_SEED}"), ("Logistic Regression", "logreg")]:
    vals, rej = bootstrap_ci(y_test.values, locked[f"{key}_prob"].values)
    boot_store[label] = vals
    point = P.compute_metrics(y_test, locked[f"{key}_prob"])
    for j, met in enumerate(["ROC_AUC", "AP", "F1"]):
        lo, hi = np.percentile(vals[:, j], [2.5, 97.5])
        ci_rows.append({"model": label, "metric": met, "point_estimate": point[met], "ci95_low": lo, "ci95_high": hi,
                        "n_boot": len(vals), "rejected_single_class": rej, "seed": P.SEEDS["bootstrap"]})
ci = pd.DataFrame(ci_rows); ci.to_csv(P.RESULTS / "test_bootstrap_ci.csv", index=False)
ci.round(4)

### Karışıklık matrisleri, ROC ve Precision–Recall eğrileri

In [ ]:
main_label, main_key = list(models)[0], f"mlp_seed{PRIMARY_SEED}"
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, (label, key) in zip(axes, [(f"MLP {FINAL_CODE} (seed {PRIMARY_SEED})", main_key), ("Logistic Regression", "logreg")]):
    r = P.compute_metrics(y_test, locked[f"{key}_prob"])
    cm = np.array([[r["TN"], r["FP"]], [r["FN"], r["TP"]]])
    ax.imshow(cm, cmap="Blues")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=14, color="white" if cm[i, j] > cm.max()/2 else "black")
    ax.set_xticks([0, 1]); ax.set_xticklabels(["Tahmin: no (0)", "Tahmin: yes (1)"])
    ax.set_yticks([0, 1]); ax.set_yticklabels(["Gerçek: no (0)", "Gerçek: yes (1)"])
    ax.set_title(f"{label}\neşik 0.50", fontsize=10)
fig.tight_layout(); fig.savefig(P.FIGURES / "fig_test_confusion_matrices.png", dpi=200); plt.close(fig)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
styles = [(f"MLP {FINAL_CODE} (seed {PRIMARY_SEED})", main_key, "#2b6cb0", "-"), ("Logistic Regression", "logreg", "black", "--"),
          ("Dummy", "dummy", "gray", ":")]
for label, key, col, ls in styles:
    p = locked[f"{key}_prob"]; r = P.compute_metrics(y_test, p)
    fpr, tpr, _ = roc_curve(y_test, p)
    axes[0].plot(fpr, tpr, color=col, ls=ls, label=f"{label} (ROC-AUC {r['ROC_AUC']:.3f})")
    prec, rec, _ = precision_recall_curve(y_test, p)
    axes[1].step(rec, prec, where="post", color=col, ls=ls, label=f"{label} (AP {r['AP']:.3f})")
axes[0].plot([0, 1], [0, 1], color="lightgray", lw=1); axes[0].set_xlabel("1 − Özgüllük (FPR)"); axes[0].set_ylabel("Duyarlılık (TPR)")
axes[0].set_title("ROC eğrisi — bağımsız test"); axes[0].legend(fontsize=8, loc="lower right")
axes[1].axhline(y_test.mean(), color="lightgray", lw=1); axes[1].set_xlabel("Recall (duyarlılık)"); axes[1].set_ylabel("Precision")
axes[1].set_title("Precision–Recall eğrisi — bağımsız test"); axes[1].legend(fontsize=8, loc="upper right")
fig.tight_layout(); fig.savefig(P.FIGURES / "fig_test_roc_pr.png", dpi=200); plt.close(fig)

### Hata analizi: False Positive ve False Negative örnekleri

In [ ]:
raw = P.load_raw().drop(columns=["duration"])
df = raw.loc[locked.index].copy()
df["y_true"] = y_test.values; df["prob"] = locked[f"{main_key}_prob"].values; df["pred"] = locked[f"{main_key}_pred"].values
df["outcome"] = np.select([(df.y_true == 1) & (df.pred == 1), (df.y_true == 0) & (df.pred == 0),
                           (df.y_true == 0) & (df.pred == 1), (df.y_true == 1) & (df.pred == 0)], ["TP", "TN", "FP", "FN"])
counts = df.outcome.value_counts().reindex(["TP", "FN", "FP", "TN"])
print("Sonuç sayıları:", counts.to_dict(), "| test pozitif:", int(y_test.sum()), "negatif:", int((1 - y_test).sum()))
err_num = df.groupby("outcome")[["prob", "age", "campaign", "previous", "euribor3m", "nr.employed"]].median().reindex(["TP", "FN", "FP", "TN"])
err_num.insert(0, "n", counts.values)
err_num["pdays_999_pct"] = df.groupby("outcome").apply(lambda g: 100 * (g.pdays == 999).mean(), include_groups=False)
err_num["poutcome_success_pct"] = df.groupby("outcome").apply(lambda g: 100 * (g.poutcome == "success").mean(), include_groups=False)
err_num["contact_cellular_pct"] = df.groupby("outcome").apply(lambda g: 100 * (g.contact == "cellular").mean(), include_groups=False)
err_num.to_csv(P.RESULTS / "test_error_analysis.csv")
display(err_num.round(3))
fn_prob = df.loc[df.outcome == "FN", "prob"]
print("FN olasılık dağılımı: medyan %.3f; %%25'i ≤ %.3f; 0.30-0.50 arası: %d"
      % (fn_prob.median(), fn_prob.quantile(.25), int(((fn_prob >= .3) & (fn_prob < .5)).sum())))
fig, ax = plt.subplots(figsize=(7, 3.8))
for lab, col in [(0, "#7f8c9a"), (1, "#2b6cb0")]:
    ax.hist(df.loc[df.y_true == lab, "prob"], bins=30, range=(0, 1), alpha=.7, color=col, label=f"Gerçek sınıf {lab}")
ax.axvline(.5, color="red", ls="--", lw=1, label="Eşik 0.50"); ax.set_yscale("log")
ax.set_xlabel("Tahmin edilen olasılık (ana MLP)"); ax.set_ylabel("Sayı (log)"); ax.legend(fontsize=8)
ax.set_title("Test olasılıklarının gerçek sınıfa göre dağılımı")
fig.tight_layout(); fig.savefig(P.FIGURES / "fig_test_probability_hist.png", dpi=200); plt.close(fig)

### Nihai modelin eğitim eğrileri ve mimari şeması

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 6.5), sharex="col")
for j, s in enumerate(WEIGHT_SEEDS):
    h = hist[hist.weight_seed == s]; inf = final_info.set_index("weight_seed").loc[s]
    axes[0, j].plot(h.epoch, h.train_loss, label="Eğitim (fit) kaybı"); axes[0, j].plot(h.epoch, h.monitor_loss, label="İzleme kaybı")
    axes[0, j].axvline(inf.best_epoch, color="gray", ls="--", lw=1)
    axes[0, j].set_title(f"seed {s}: en iyi epoch {inf.best_epoch}, durma {inf.stopped_epoch}", fontsize=10)
    axes[1, j].plot(h.epoch, h.lr, color="#c05621"); axes[1, j].set_yscale("log"); axes[1, j].set_xlabel("Epoch")
axes[0, 0].set_ylabel("BCE kaybı"); axes[1, 0].set_ylabel("Öğrenme oranı"); axes[0, 0].legend(fontsize=8)
fig.suptitle(f"Nihai model {FINAL_CODE}: eğitim ve erken durdurma izleme kaybı, epoch bazında öğrenme oranı")
fig.tight_layout(); fig.savefig(P.FIGURES / "fig_training_curves_final.png", dpi=200); plt.close(fig)

n_in = pm.info["n_inputs"]; layers = [n_in] + list(FINAL["hidden_layers"]) + [1]
parts, prev = [], n_in
for h_ in list(FINAL["hidden_layers"]) + [1]:
    parts.append(f"{prev}×{h_} + {h_} = {prev*h_ + h_}"); prev = h_
total = P.n_params_formula(n_in, FINAL["hidden_layers"]); assert total == pm.info["n_params"]
fig, ax = plt.subplots(figsize=(11, 3.6)); ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")
names = ["Girdi\n(ön işlenmiş)"] + [f"Gizli katman {i+1}\nLinear → {FINAL['activation']}" + (f"\n→ Dropout {FINAL['dropout']}" if FINAL['dropout'] else "") for i in range(len(FINAL["hidden_layers"]))] + ["Çıkış\nLinear → 1 logit"]
xs = np.linspace(.03, .78, len(layers)); w = .16
for i, (x, n, name) in enumerate(zip(xs, layers, names)):
    ax.add_patch(FancyBboxPatch((x, .35), w, .45, boxstyle="round,pad=0.01", fc="#eef2f7", ec="#4a5568"))
    ax.text(x + w/2, .575, f"{name}\n{n} birim", ha="center", va="center", fontsize=9)
    if i > 0:
        ax.annotate("", xy=(x, .575), xytext=(xs[i-1] + w, .575), arrowprops=dict(arrowstyle="-|>"))
        ax.text((x + xs[i-1] + w)/2, .2, parts[i-1], ha="center", fontsize=8)
ax.add_patch(FancyBboxPatch((.84, .55), .15, .25, boxstyle="round,pad=0.01", fc="#f7f1ee", ec="#4a5568"))
ax.text(.915, .675, "Eğitim:\nBCEWithLogitsLoss", ha="center", va="center", fontsize=8)
ax.add_patch(FancyBboxPatch((.84, .2), .15, .25, boxstyle="round,pad=0.01", fc="#eef7f3", ec="#4a5568"))
ax.text(.915, .325, "Tahmin: sigmoid\n→ olasılık ≥ 0.50", ha="center", va="center", fontsize=8)
ax.annotate("", xy=(.84, .675), xytext=(xs[-1] + w, .6), arrowprops=dict(arrowstyle="-|>"))
ax.annotate("", xy=(.84, .325), xytext=(xs[-1] + w, .55), arrowprops=dict(arrowstyle="-|>"))
ax.text(.42, .05, f"Toplam öğrenilebilir parametre (ağırlık + bias): {total}", ha="center", fontsize=10, fontweight="bold")
ax.set_title(f"Nihai MLP ({FINAL_CODE}) mimarisi; her katman için parametre = girdi × çıktı + çıktı", fontsize=10)
fig.tight_layout(); fig.savefig(P.FIGURES / "fig_mlp_architecture.png", dpi=200); plt.close(fig)
print("Parametre dökümü:", parts, "toplam:", total)

### Dış CV, development iç CV ve bağımsız test sonuçlarının yan yana gösterimi

In [ ]:
outer = pd.read_csv(P.RESULTS / "outer_cv_results.csv")
cmp_rows = []
for meth in ["Random search", "Manual search", "LogReg", "Dummy"]:
    o = outer[outer.method == meth]
    cmp_rows.append({"source": f"Dış CV ({meth})", "AP": P.mean_sd(o.AP), "ROC_AUC": P.mean_sd(o.ROC_AUC), "F1": P.mean_sd(o.F1)})
cmp_rows.append({"source": f"Development iç CV ({FINAL_CODE}, seçim aşaması)",
                 "AP": f"{sel['dev_inner_cv_mean_AP'][FINAL_CODE]:.4f} ± {sel['dev_inner_cv_sd_AP'][FINAL_CODE]:.4f}", "ROC_AUC": "-", "F1": "-"})
for lab in [list(models)[0], "Logistic Regression", "Dummy (most_frequent)"]:
    t = test_metrics.loc[lab]
    cmp_rows.append({"source": f"Bağımsız test: {lab}", "AP": f"{t.AP:.4f}", "ROC_AUC": f"{t.ROC_AUC:.4f}", "F1": f"{t.F1:.4f}"})
cmp = pd.DataFrame(cmp_rows); cmp.to_csv(P.RESULTS / "results_comparison_outer_inner_test.csv", index=False)
cmp